# Read and plot the original source

Same complete `Fig_2B` readout and plotting options as `read_atst.ipynb`. Source ranges and condition annotations remain explicit; no ATST file is read. Matplotlib plots the source values directly.


In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd
from openpyxl import load_workbook

readout_id = 'Fig_2B'
rows, curves = [], []

sheet = load_workbook("read_shared_data/sources/aem.02402-24-s0002.xlsx", read_only=True, data_only=True)["Fig 2B"]
time = [row[0].value for row in sheet["A2:A11"]]
for phage, signals, first in [("", "", 2), ("", "3OC12-HSL 0.5 uM + C4-HSL 0.4 uM", 5), ("phiPA2", "", 8), ("phiPA2", "3OC12-HSL 0.5 uM + C4-HSL 0.4 uM", 11)]:
    for replicate, column in enumerate(range(first, first + 3), 1):
        rows.append(dict(type="phage_exposed" if phage else "uninfected_control", phage_id=phage, moi="0.01" if phage else "0", signaling_molecules=signals, replicate=str(replicate)))
        curves.append(pd.Series([sheet.cell(r, column).value for r in range(2, 12)], index=time))

# Local curve IDs preserve source order; different time grids align without interpolation.
layout = pd.DataFrame(rows)
layout["curve_id"] = [f"curve_{i}" for i in range(1, len(rows) + 1)]
readings = pd.concat(curves, axis=1).sort_index()
readings.columns = layout["curve_id"]
readings = readings.rename_axis("Time").reset_index()


In [ ]:
panels = list(layout.groupby('signaling_molecules', sort=False))
ncols = min(2, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = {}

for ax, (panel, panel_layout) in zip(axes.flat, panels):
    for condition, group in panel_layout.groupby(['phage_id', 'moi'], sort=False):
        if condition not in condition_colors:
            condition_colors[condition] = palette[len(condition_colors) % len(palette)]
        color = condition_colors[condition]
        label = " | ".join(str(value) for value in condition)
        values = readings[group["curve_id"]]
        for index, curve_id in enumerate(group["curve_id"]):
            ax.plot(
                readings["Time"], readings[curve_id], color=color, alpha=0.6,
                label=label if index == 0 else None,
            )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("od600")
    panel_label = "<blank>" if panel == "" else str(panel)
    ax.set_title(f"{readout_id} | signaling_molecules={panel_label}")
    ax.legend(frameon=False)

for ax in list(axes.flat)[len(panels):]:
    fig.delaxes(ax)
fig.tight_layout()
plt.show()


In [ ]:
panels = list(layout.groupby('signaling_molecules', sort=False))
ncols = min(2, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = {}

for ax, (panel, panel_layout) in zip(axes.flat, panels):
    for condition, group in panel_layout.groupby(['phage_id', 'moi'], sort=False):
        if condition not in condition_colors:
            condition_colors[condition] = palette[len(condition_colors) % len(palette)]
        color = condition_colors[condition]
        label = " | ".join(str(value) for value in condition)
        values = readings[group["curve_id"]]
        median = values.median(axis=1)
        minimum = values.min(axis=1)
        maximum = values.max(axis=1)
        ax.errorbar(
            readings["Time"], median,
            yerr=[median - minimum, maximum - median],
            fmt="o-", capsize=3, color=color, label=label,
        )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("od600")
    panel_label = "<blank>" if panel == "" else str(panel)
    ax.set_title(f"{readout_id} | signaling_molecules={panel_label}")
    ax.legend(frameon=False)

for ax in list(axes.flat)[len(panels):]:
    fig.delaxes(ax)
fig.tight_layout()
plt.show()
